# Costruzione matrice per lavorare

Questo programma servirà esclusivamente per costruire e poi salvare la matrice originale "matrice_con_t.dat".
La matrice è fondamentale per runnare e studiare la convoluzione con la svd, quindi questo è il primo passo di preprocessing.
Verranno solo uniti gli spettri di tutte le temperature in solo 1.

In [1]:
import pandas as pd
from pathlib import Path

# Percorso relativo al file corrente (se script)
try:
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Se sei in Jupyter
    base_dir = Path().resolve()

folder_path = base_dir / "data_cd_cMYC"

file_list = list(folder_path.glob("*.txt"))

dfs = []

for file_path in file_list:
    with open(file_path, 'r') as file:
        lines = file.readlines()

    start_idx = next(i for i, line in enumerate(lines) if "XYDATA" in line) + 1

    df = pd.read_csv(
        file_path,
        sep="\t",
        skiprows=start_idx,
        header=None,
        usecols=[0, 1],
        names=["Wavelength", file_path.stem]
    )

    dfs.append(df)
merged_df = dfs[0]

for df in dfs[1:]:  
    merged_df = pd.merge(merged_df, df, on= "Wavelength", how='inner')

merged_df.head()

,Wavelength,CD_cmyc_45uM_T88,CD_cmyc_45uM_T76,CD_cmyc_45uM_T62,CD_cmyc_45uM_T48,CD_cmyc_45uM_T74,CD_cmyc_45uM_T60,CD_cmyc_45uM_T58,CD_cmyc_45uM_T64,CD_cmyc_45uM_T70,...,CD_cmyc_45uM_T50,CD_cmyc_45uM_T44,CD_cmyc_45uM_T78,CD_cmyc_45uM_T86,CD_cmyc_45uM_T92,CD_cmyc_45uM_T84,CD_cmyc_45uM_T90,CD_cmyc_45uM_T52,CD_cmyc_45uM_T46,CD_cmyc_45uM_T100
0,330.0,0.449928,0.489877,0.473675,0.584630,0.453920,0.508665,0.500564,0.462383,0.452789,...,0.549599,0.598129,0.493656,0.412534,0.416314,0.474593,0.433836,0.580404,0.521206,0.359595
1,329.5,0.454074,0.485369,0.476797,0.577216,0.448509,0.502561,0.511669,0.421921,0.437550,...,0.537235,0.598848,0.484736,0.424608,0.443013,0.474701,0.441275,0.575467,0.519017,0.364532
2,329.0,0.445359,0.506257,0.480524,0.585907,0.441575,0.511338,0.512455,0.411955,0.456915,...,0.543890,0.592043,0.482867,0.426925,0.440004,0.470591,0.448967,0.566430,0.531511,0.359207
3,328.5,0.425127,0.514101,0.466812,0.586902,0.429242,0.506663,0.521813,0.409982,0.469810,...,0.543401,0.583251,0.480349,0.441010,0.448670,0.479963,0.451567,0.563792,0.536599,0.375066
4,328.0,0.417484,0.507298,0.471372,0.589668,0.431725,0.511624,0.534050,0.408896,0.472435,...,0.537625,0.595377,0.485958,0.455050,0.456393,0.487527,0.442674,0.565210,0.530009,0.381205


Ora è necessario scrivere il df in modo leggibile per il software successivo

In [2]:
# 1. Estrai le temperature dai nomi delle colonne (escludendo "Wavelength")
temp_dict = {
    col: int(col.split('_T')[-1])
    for col in merged_df.columns if col != "Wavelength"
}

# 2. Ordina i nomi delle colonne in base alla temperatura
sorted_cols = sorted(temp_dict, key=lambda col: temp_dict[col])

# 3. Ricostruisci l'ordine finale con "Wavelength" all'inizio
final_columns = ["Wavelength"] + sorted_cols
merged_df_sorted = merged_df[final_columns]

# 4. Costruisci la riga con le temperature (metti 'Wavelength' come primo valore)
temperature_row = ['Wavelength'] + [temp_dict[col] for col in sorted_cols]

# 5. Crea DataFrame con la riga delle temperature e concatena
temp_df = pd.DataFrame([temperature_row], columns=merged_df_sorted.columns)
merged_df_final = pd.concat([temp_df, merged_df_sorted], ignore_index=True)
merged_df_final.reset_index(drop=True, inplace=True)
merged_df_final.head()

,Wavelength,CD_cmyc_45uM_T20,CD_cmyc_45uM_T22,CD_cmyc_45uM_T24,CD_cmyc_45uM_T26,CD_cmyc_45uM_T28,CD_cmyc_45uM_T30,CD_cmyc_45uM_T32,CD_cmyc_45uM_T34,CD_cmyc_45uM_T36,...,CD_cmyc_45uM_T82,CD_cmyc_45uM_T84,CD_cmyc_45uM_T86,CD_cmyc_45uM_T88,CD_cmyc_45uM_T90,CD_cmyc_45uM_T92,CD_cmyc_45uM_T94,CD_cmyc_45uM_T96,CD_cmyc_45uM_T98,CD_cmyc_45uM_T100
0,Wavelength,20.000000,22.000000,24.000000,26.000000,28.000000,30.000000,32.000000,34.000000,36.000000,...,82.000000,84.000000,86.000000,88.000000,90.000000,92.000000,94.000000,96.000000,98.000000,100.000000
1,330.0,0.690073,0.594515,0.558850,0.539230,0.518909,0.572465,0.529252,0.535038,0.553952,...,0.482802,0.474593,0.412534,0.449928,0.433836,0.416314,0.422019,0.356426,0.416018,0.359595
2,329.5,0.690435,0.605249,0.545716,0.544413,0.530282,0.549292,0.512822,0.514569,0.552011,...,0.480866,0.474701,0.424608,0.454074,0.441275,0.443013,0.404459,0.364644,0.390242,0.364532
3,329.0,0.697104,0.613594,0.539834,0.542031,0.552200,0.523191,0.511314,0.504031,0.539964,...,0.471653,0.470591,0.426925,0.445359,0.448967,0.440004,0.402753,0.397843,0.378232,0.359207
4,328.5,0.687360,0.599005,0.534644,0.524888,0.520019,0.512783,0.497171,0.504198,0.544028,...,0.476281,0.479963,0.441010,0.425127,0.451567,0.448670,0.423389,0.416523,0.403924,0.375066


Ora, lo salviamo!

In [3]:
# Percorso file
output_file = base_dir / "matrice_con_t.dat"

if not output_file.exists():
    merged_df_final.to_csv(output_file, index=False, header=False)

print(f"File salvato come '{output_file.name}'")


File salvato come 'matrice_con_t.dat'
